In [1]:
import os
import json
from pathlib import Path
from tqdm import tqdm

import pandas as pd 

from custom_tools.visualization_tools import read_file

Формирование фрейма с метаинформацией о датасете

In [ ]:
# Директория с отобранными видео пациентов
PATH_SELECTED_VIDEO = Path("../../datasets/CADICA/selectedVideos")
# Файл с описанием какое видео относится к LCA и RCA
CADICAprojections_path = "../../datasets/CADICA/CADICAprojections.json"

Все отобранные видео пациентов (с поражениями, без поражений)

In [ ]:
records = []

all_patients = sorted([p for p in os.listdir(PATH_SELECTED_VIDEO) if os.path.isdir(PATH_SELECTED_VIDEO / p)])

print(f"Найдено пациентов: {len(all_patients)}")

for px in tqdm(all_patients, desc="Обработка пациентов"):
    patient_path = PATH_SELECTED_VIDEO / px
    
    # Читаем списки видео
    lesion_videos = read_file(patient_path / "lesionVideos.txt")
    nonlesion_videos = read_file(patient_path / "nonlesionVideos.txt")
    
    # === Обработка видео с поражениями (label = 1) ===
    for vx in lesion_videos:
        video_path = patient_path / vx
        frames_txt = video_path / f"{px}_{vx}_selectedFrames.txt"
        gt_dir = video_path / "groundtruth"
        
        selected_frames = read_file(frames_txt)
        
        for frame_name in selected_frames:
            img_path = video_path / "input" / f"{frame_name}.png"
            gt_path = gt_dir / f"{frame_name}.txt"
            
            if not img_path.exists():
                continue
                
            # Проверяем, действительно ли есть поражение
            has_lesion = False
            num_bboxes = 0
            if gt_path.exists():
                bboxes = read_file(gt_path)
                num_bboxes = len(bboxes)
                has_lesion = num_bboxes > 0
            
            records.append({
                'patient': px,
                'video': vx,
                'frame': frame_name,
                'image_path': str(img_path),
                'label': 1 if has_lesion else 0,
                'num_bboxes': num_bboxes
            })
    
    # === Обработка видео без поражений (label = 0) ===
    for vx in nonlesion_videos:
        video_path = patient_path / vx
        frames_txt = video_path / f"{px}_{vx}_selectedFrames.txt"
        
        selected_frames = read_file(frames_txt)
        
        for frame_name in selected_frames:
            img_path = video_path / "input" / f"{frame_name}.png"
            
            if not img_path.exists():
                continue
                
            records.append({
                'patient': px,
                'video': vx,
                'frame': frame_name,
                'image_path': str(img_path),
                'label': 0,
                'num_bboxes': 0
            })

# ====================== Создание DataFrame ======================
df = pd.DataFrame(records)

Найдено пациентов: 42


Обработка пациентов:   0%|          | 0/42 [00:00<?, ?it/s]

Обработка пациентов: 100%|██████████| 42/42 [00:27<00:00,  1.55it/s]


In [6]:
df

,patient,video,frame,image_path,label,num_bboxes
0,p1,v2,p1_v2_00015,..\..\datasets\CADICA\selectedVideos\p1\v2\inp...,1,1
1,p1,v2,p1_v2_00016,..\..\datasets\CADICA\selectedVideos\p1\v2\inp...,1,1
2,p1,v2,p1_v2_00017,..\..\datasets\CADICA\selectedVideos\p1\v2\inp...,1,1
3,p1,v2,p1_v2_00018,..\..\datasets\CADICA\selectedVideos\p1\v2\inp...,1,1
4,p1,v2,p1_v2_00019,..\..\datasets\CADICA\selectedVideos\p1\v2\inp...,1,1
...,...,...,...,...,...,...
6121,p9,v9,p9_v9_00037,..\..\datasets\CADICA\selectedVideos\p9\v9\inp...,0,0
6122,p9,v9,p9_v9_00038,..\..\datasets\CADICA\selectedVideos\p9\v9\inp...,0,0
6123,p9,v9,p9_v9_00039,..\..\datasets\CADICA\selectedVideos\p9\v9\inp...,0,0
6124,p9,v9,p9_v9_00040,..\..\datasets\CADICA\selectedVideos\p9\v9\inp...,0,0


Добавляем "ID" для видео (**patient**, **video**). 

Из файла CADICAprojections.json. Определяем к какому виду коронарной артерии относится (**LCA** - левая, **RCA** - правая)

In [ ]:
# Добавить id для видео
df["video_id"] = df['patient'] + "_" + df["video"]

# === add LCA or RCA informations ===
with open(CADICAprojections_path, "r") as file:
    rca_lca_data = json.load(file)

# rca_lca_data['videosLCA2'] - еще есть около 30 видео с LCA
lca_videos = set(rca_lca_data['videosLCA'])
rca_videos = set(rca_lca_data["videosRCA"])

df["artery"] = df["video_id"].apply(lambda x: "RCA" if x in rca_videos else ("LCA" if x in lca_videos else "Unknown"))

print(df.shape)
print(df.groupby("artery")["video_id"].nunique())

(6126, 8)
artery
LCA        216
RCA        118
Unknown     48
Name: video_id, dtype: int64


Полученное количество видео для LCA и RCA, а также общее количество фреймов соответствует заявленному количеству в статье

Сохраняем полученный фрейм

In [ ]:
df.to_csv("../data/cadica_dataset.csv", index=False)

In [ ]:
print("\n=== ИТОГОВАЯ СТАТИСТИКА ===")
print(f"Всего кадров:               {df.shape[0]:,}")
print(f"С поражением (label=1):     {df['label'].sum():,}")
print(f"Без поражения (label=0):    {(df['label'] == 0).sum():,}")
print(f"Баланс Lesion:              {df['label'].mean():.4f} ({df['label'].mean()*100:.1f}%)")

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6126 entries, 0 to 6125
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   patient     6126 non-null   object
 1   video       6126 non-null   object
 2   frame       6126 non-null   object
 3   image_path  6126 non-null   object
 4   label       6126 non-null   int64 
 5   num_bboxes  6126 non-null   int64 
 6   video_id    6126 non-null   object
 7   artery      6126 non-null   object
dtypes: int64(2), object(6)
memory usage: 383.0+ KB


In [10]:
df

,patient,video,frame,image_path,label,num_bboxes,video_id,artery
0,p1,v2,p1_v2_00015,..\..\datasets\CADICA\selectedVideos\p1\v2\inp...,1,1,p1_v2,LCA
1,p1,v2,p1_v2_00016,..\..\datasets\CADICA\selectedVideos\p1\v2\inp...,1,1,p1_v2,LCA
2,p1,v2,p1_v2_00017,..\..\datasets\CADICA\selectedVideos\p1\v2\inp...,1,1,p1_v2,LCA
3,p1,v2,p1_v2_00018,..\..\datasets\CADICA\selectedVideos\p1\v2\inp...,1,1,p1_v2,LCA
4,p1,v2,p1_v2_00019,..\..\datasets\CADICA\selectedVideos\p1\v2\inp...,1,1,p1_v2,LCA
...,...,...,...,...,...,...,...,...
6121,p9,v9,p9_v9_00037,..\..\datasets\CADICA\selectedVideos\p9\v9\inp...,0,0,p9_v9,Unknown
6122,p9,v9,p9_v9_00038,..\..\datasets\CADICA\selectedVideos\p9\v9\inp...,0,0,p9_v9,Unknown
6123,p9,v9,p9_v9_00039,..\..\datasets\CADICA\selectedVideos\p9\v9\inp...,0,0,p9_v9,Unknown
6124,p9,v9,p9_v9_00040,..\..\datasets\CADICA\selectedVideos\p9\v9\inp...,0,0,p9_v9,Unknown


In [12]:
df[(df['patient'] == "p10") & (df['video'] == "v6") & (df['label'] == 1)]

,patient,video,frame,image_path,label,num_bboxes,video_id,artery
325,p10,v6,p10_v6_00013,..\..\datasets\CADICA\selectedVideos\p10\v6\in...,1,1,p10_v6,RCA
326,p10,v6,p10_v6_00014,..\..\datasets\CADICA\selectedVideos\p10\v6\in...,1,2,p10_v6,RCA
327,p10,v6,p10_v6_00015,..\..\datasets\CADICA\selectedVideos\p10\v6\in...,1,1,p10_v6,RCA
328,p10,v6,p10_v6_00016,..\..\datasets\CADICA\selectedVideos\p10\v6\in...,1,2,p10_v6,RCA
329,p10,v6,p10_v6_00017,..\..\datasets\CADICA\selectedVideos\p10\v6\in...,1,1,p10_v6,RCA
330,p10,v6,p10_v6_00018,..\..\datasets\CADICA\selectedVideos\p10\v6\in...,1,1,p10_v6,RCA
331,p10,v6,p10_v6_00019,..\..\datasets\CADICA\selectedVideos\p10\v6\in...,1,1,p10_v6,RCA
332,p10,v6,p10_v6_00020,..\..\datasets\CADICA\selectedVideos\p10\v6\in...,1,1,p10_v6,RCA
333,p10,v6,p10_v6_00021,..\..\datasets\CADICA\selectedVideos\p10\v6\in...,1,2,p10_v6,RCA
334,p10,v6,p10_v6_00022,..\..\datasets\CADICA\selectedVideos\p10\v6\in...,1,2,p10_v6,RCA
